# 108 — DS5 Balanced Profile Radar (Level 1 Visualization Fix)

**Purpose:** Reproduce the DS5 phenotype radar chart (`fig4_fontphenotype_profiles_enhanced_notitle`)
with a **balanced 5-axis configuration** — exactly one feature per CAS model family — using the
identical plot functions (individual lines, IQR bands, section-layer bands, arc labels, label boxes).

**NOT:** a new analysis, a new clustering, or a modification of any result.
**IS:** the same DS5 Phase 1 cluster assignments + a different axis selection.

## Why notebook 107 was wrong
Notebook 107 used a simplified `plot_balanced_radar()` function — a basic polar fill with no
individual participant lines, no IQR bands, and no section-layer arc decorations.
The user-requested style (matching the Structured dataset output) requires the full
`plot_single_profile()` + `draw_model_arcs()` + `draw_section_layers()` + `draw_axis_label_boxes()`
pipeline from notebook 100 cell G06b.

## Axis selection (one per family)

| Axis label | Feature | Source | Model family | Why |
|---|---|---|---|---|
| Behavioral entropy | `H_emp` | participant_dynamics_parameters.csv | InT | entropy of daily override-rate distribution |
| Impulsivity (noise) | `sig_hat` | participant_dynamics_parameters.csv | StoP | diffusion σ̂ (behavioral volatility) |
| Dynamical deviation | `deviation_from_predicted_fixed_point` | participant_dynamics_parameters.csv | DyS | distance from SDE-predicted fixed point |
| Network concentration | `HHI_net` | participant_network_metrics.csv | NeT | HHI of app-transition flows |
| Susceptibility | `chi_behav` | participant_stm_parameters.csv | StM | χ = β·p·(1−p), peaks near threshold |

**Key difference from original fig4:** original had 7 axes (2×NeT, 2×StoP, 2×InT, 1×StM, 0×DyS);
this has 5 axes (1×InT, 1×StoP, 1×DyS, 1×NeT, 1×StM).

In [ ]:
from collections import OrderedDict
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

In [ ]:
ROOT        = Path('..')
MODEL_OUT   = ROOT / 'outputs/model_outputs'
PHENO_OUT   = MODEL_OUT / 'phenotyping'
FIG_OUT     = PHENO_OUT / 'figures'
FIG_OUT.mkdir(parents=True, exist_ok=True)

In [ ]:
# Load Phase 1 cluster assignments
cl = pd.read_csv(PHENO_OUT / 'ds5_cluster_assignments.csv')
p1 = cl[['participant_id_canonical', 'profile_p1']].copy()
print(f'Phase 1 participants: {len(p1)}')
print(p1['profile_p1'].value_counts())

In [ ]:
# Load feature files — all three InT/StoP/DyS features come from the DyS parameter file
dys = pd.read_csv(MODEL_OUT / 'dynamical_systems/participant_dynamics_parameters.csv',
                  usecols=['participant_id_canonical', 'H_emp', 'sig_hat',
                           'deviation_from_predicted_fixed_point'])

net = pd.read_csv(MODEL_OUT / 'network_models/participant_network_metrics.csv',
                  usecols=['participant_id_canonical', 'HHI_net'])

stm = pd.read_csv(MODEL_OUT / 'statistical_mechanics/participant_stm_parameters.csv',
                  usecols=['participant_id_canonical', 'chi_behav'])

print('DyS features:', dys.shape, '  NaN:', dys.isna().sum().to_dict())
print('NeT features:', net.shape, '  NaN:', net.isna().sum().to_dict())
print('StM features:', stm.shape, '  NaN:', stm.isna().sum().to_dict())

In [ ]:
# Merge into p1_pop — all features must join onto Phase 1 participants
p1_pop = (p1
          .merge(dys, on='participant_id_canonical', how='left')
          .merge(net, on='participant_id_canonical', how='left')
          .merge(stm, on='participant_id_canonical', how='left'))

print(f'p1_pop shape: {p1_pop.shape}')
print('NaN counts:')
print(p1_pop.isna().sum())
print()
print('Profile A:', len(p1_pop[p1_pop['profile_p1']=='Profile A']))
print('Profile B:', len(p1_pop[p1_pop['profile_p1']=='Profile B']))

In [ ]:
# ─────────────────────────────────────────────
# CONFIGURATION — balanced 5-axis selection
# One feature per CAS model family.
# Axis order: InT / StoP / DyS grouped as 'Behavioral dynamics',
#             NeT as 'Network structure', StM as 'Thermodynamic'.
# ─────────────────────────────────────────────

RADAR_DIMS = OrderedDict([
    ('Behavioral\nentropy',    'H_emp'),                              # InT
    ('Impulsivity\n(noise)',   'sig_hat'),                            # StoP
    ('Dynamical\ndeviation',   'deviation_from_predicted_fixed_point'), # DyS
    ('Network\nconcentration', 'HHI_net'),                            # NeT
    ('Susceptibility\n(percentile)', 'chi_behav'),                    # StM (rank-normalized)
])

AXIS_MODEL = {
    'Behavioral\nentropy':    'InT',
    'Impulsivity\n(noise)':   'StoP',
    'Dynamical\ndeviation':   'DyS',
    'Network\nconcentration': 'NeT',
    'Susceptibility\n(percentile)': 'StM',
}

MODEL_COLORS = {
    'InT':  '#2196F3',   # blue
    'StoP': '#FF9800',   # orange
    'DyS':  '#9C27B0',   # purple
    'NeT':  '#4CAF50',   # green
    'StM':  '#C5D307',   # yellow-green
}

PROFILE_COLORS = {'Profile A': '#A20A7C', 'Profile B': '#7E57C2'}

# Section layers — grouped by conceptual theme, indices into RADAR_DIMS order
SECTION_LAYERS = [
    {'name': 'Behavioral dynamics', 'dim_idx': [0, 1, 2], 'color': '#C62828'},  # InT+StoP+DyS
    {'name': 'Network structure',   'dim_idx': [3],       'color': '#044B07'},  # NeT
    {'name': 'Thermodynamic',       'dim_idx': [4],       'color': '#0A4B96'},  # StM
]

FONT_SIZES = {
    'figure_title':     20.0,
    'subplot_title':    1.0,
    'axis_labels':      16,
    'section_layers':   16,
    'grid_ticks':       13,
    'annotations':      13,
    'subplot_legend':   16,
    'bottom_legends':   16,
    'legend_titles':    16,
}

In [ ]:
# ─────────────────────────────────────────────
# HELPER FUNCTIONS
# Copied verbatim from notebook 100 cell G06b (the version that produced
# fig4_fontphenotype_profiles_enhanced_notitle.png). No modifications.
# ─────────────────────────────────────────────

# Heavy-tailed axes: min-max scaling puts every group mean on the origin (chi_behav
# median 9.87 vs max 266.84 -> profile means 0.085 / 0.055), hiding a real difference
# (Mann-Whitney U on raw chi_behav, Profile A vs B: p=0.0020). Percentile-rank these.
RANK_NORMALIZED_FEATS = {'chi_behav'}


def normalize_features(df, dim_feats):
    """Min-max normalize each feature across the full Phase 1 population.

    Features in RANK_NORMALIZED_FEATS are percentile-ranked instead.
    """
    norm = {}
    for feat in dim_feats:
        if feat in RANK_NORMALIZED_FEATS:
            norm[feat] = df[feat].rank(pct=True)
            continue
        v = df[feat].dropna()
        norm[feat] = (df[feat] - v.min()) / (v.max() - v.min() + 1e-12)
    return norm


def draw_model_arcs(ax, angles, dim_labels, axis_model, model_colors,
                    arc_r=1.18, arc_lw=6, label_r=1.34, fontsize=9):
    """Colored arc segment per axis with grouped family label in the arc ring."""
    n = len(angles)
    half_gap = (2 * np.pi / n) * 0.38
    family_angles = {}

    for label, angle in zip(dim_labels, angles):
        model = axis_model.get(label, '')
        color = model_colors.get(model, '#AAAAAA')
        theta = np.linspace(angle - half_gap, angle + half_gap, 40)
        ax.plot(theta, np.full_like(theta, arc_r),
                color=color, lw=arc_lw, solid_capstyle='round',
                clip_on=False, zorder=10)
        family_angles.setdefault(model, []).append(angle)

    for model, ang_list in family_angles.items():
        mid = np.arctan2(np.mean(np.sin(ang_list)), np.mean(np.cos(ang_list)))
        color = model_colors.get(model, '#AAAAAA')
        ax.text(mid, label_r, model,
                ha='center', va='center', fontsize=fontsize, fontweight='bold',
                color=color, clip_on=False, zorder=11,
                bbox=dict(boxstyle='round,pad=0.2', fc='white',
                          ec=color, lw=1.2, alpha=0.90))


def draw_section_layers(ax, angles, section_layers,
                        band_inner=1.09, band_outer=1.20, label_r=1.145):
    """Colored arc shadow band per conceptual layer (no text — commented out in source)."""
    step = angles[1] - angles[0]
    half = step / 2
    for sec in section_layers:
        idx   = sec['dim_idx']
        color = sec['color']
        a0    = angles[idx[0]]  - half
        a1    = angles[idx[-1]] + half
        theta = np.linspace(a0, a1, 300)
        ax.fill_between(theta, band_inner, band_outer,
                        color=color, alpha=0.13, clip_on=False, zorder=5)
        ax.plot(theta, np.full_like(theta, band_inner),
                color=color, lw=0.8, alpha=0.40, clip_on=False, zorder=6)
        ax.plot(theta, np.full_like(theta, band_outer),
                color=color, lw=0.8, alpha=0.40, clip_on=False, zorder=6)


def draw_axis_label_boxes(ax, angles, dim_labels, label_r=1.33):
    """Place each axis label in a white rounded box outside the section bands."""
    for label, angle in zip(dim_labels, angles):
        ax.text(angle, label_r, label,
                ha='center', va='center',
                fontsize=FONT_SIZES['axis_labels'], color='#222222',
                clip_on=False, zorder=25, multialignment='center',
                bbox=dict(boxstyle='round,pad=0.30', fc='white',
                          ec='#BBBBBB', lw=0.9, alpha=0.97))


def compute_iqr_band(sub_df, norm_cols, angles):
    """Return closed arrays (q25, q75, mean) aligned with `angles`."""
    q25  = sub_df[norm_cols].quantile(0.25).to_numpy(dtype=float)
    q75  = sub_df[norm_cols].quantile(0.75).to_numpy(dtype=float)
    mean = sub_df[norm_cols].mean(skipna=True).to_numpy(dtype=float)
    def close(arr): return np.append(arr, arr[0])
    return close(q25), close(q75), close(mean)


def plot_single_profile(ax, sub_df, norm_cols, angles, profile_name,
                        color, dim_labels, axis_model, model_colors,
                        section_layers, individual_alpha=0.20, individual_lw=0.6,
                        mean_lw=2.8, show_annotations=False):
    """
    Draw onto polar axis:
      - thin individual participant lines
      - IQR shaded band (25th–75th percentile)
      - thick mean profile line with hollow markers
      - model-family arc segments + section layer bands + label boxes
    """
    for _, row in sub_df.iterrows():
        v = np.append(row[norm_cols].to_numpy(dtype=float), row[norm_cols].iloc[0])
        ax.plot(angles, v, color=color, alpha=individual_alpha, lw=individual_lw, zorder=1)

    q25, q75, mean_vals = compute_iqr_band(sub_df, norm_cols, angles)
    ax.fill_between(angles, q25, q75, color=color, alpha=0.22, zorder=2)
    ax.plot(angles, q25, color=color, alpha=0.45, lw=1.0, ls='--', zorder=4)
    ax.plot(angles, q75, color=color, alpha=0.45, lw=1.0, ls='--', zorder=4)

    ax.plot(angles, mean_vals, 'o-', color=color, lw=mean_lw, ms=5,
            markerfacecolor='white', markeredgecolor=color, markeredgewidth=1.8,
            label=f'{profile_name} mean  (n={len(sub_df)})', zorder=5)
    ax.fill(angles, mean_vals, color=color, alpha=0.10, zorder=3)

    draw_model_arcs(ax, angles[:-1], dim_labels, axis_model, model_colors)
    draw_section_layers(ax, angles[:-1], section_layers)
    draw_axis_label_boxes(ax, angles[:-1], dim_labels)

    ax.set_xticks(angles[:-1])
    ax.set_xticklabels([''] * len(dim_labels))
    ax.tick_params(axis='x', length=0)
    ax.set_ylim(0, 1)
    ax.set_yticks([0.25, 0.50, 0.75, 1.00])
    ax.set_yticklabels(['0.25', '0.50', '0.75', '1.00'],
                       fontsize=FONT_SIZES['grid_ticks'], alpha=0.5, color='#555555')
    ax.spines['polar'].set_color('#272727')
    ax.grid(color='#121111', lw=0.6)

In [ ]:
# ─────────────────────────────────────────────
# MAIN FIGURE FUNCTION
# ─────────────────────────────────────────────

def plot_phenotype_profiles_balanced(
    p1_pop,
    profile_col='profile_p1',
    participant_col='participant_id_canonical',
    save_path=None,
    figsize=(18, 9.5),
):
    """Side-by-side radar: Profile A (left) and Profile B (right).
    Uses the full G06b styling from notebook 100 with 5 balanced axes.
    """
    available  = OrderedDict((k, v) for k, v in RADAR_DIMS.items() if v in p1_pop.columns)
    dim_labels = list(available.keys())
    dim_feats  = list(available.values())
    am         = {k: AXIS_MODEL[k] for k in dim_labels}

    norm_map = normalize_features(p1_pop, dim_feats)
    p1_radar = p1_pop.copy()
    for feat in dim_feats:
        p1_radar[f'{feat}_norm'] = norm_map[feat]
    norm_cols = [f'{feat}_norm' for feat in dim_feats]

    angles = np.linspace(0, 2 * np.pi, len(dim_labels), endpoint=False)
    angles = np.append(angles, angles[0])

    fig, axes = plt.subplots(1, 2, figsize=figsize, subplot_kw=dict(polar=True))
    fig.subplots_adjust(wspace=0.7, left=0.08, right=0.92, top=0.85, bottom=0.12)

    specs = [
        ('Profile A', PROFILE_COLORS['Profile A'], axes[0], False),
        ('Profile B', PROFILE_COLORS['Profile B'], axes[1], True),
    ]

    for pname, color, ax, annotate in specs:
        sub = p1_radar[p1_radar[profile_col] == pname].copy()
        if sub.empty:
            ax.set_title(f'{pname}\n(no data)', fontsize=FONT_SIZES['subplot_title'])
            continue
        plot_single_profile(
            ax=ax, sub_df=sub, norm_cols=norm_cols, angles=angles,
            profile_name=pname, color=color, dim_labels=dim_labels,
            axis_model=am, model_colors=MODEL_COLORS,
            section_layers=SECTION_LAYERS, show_annotations=annotate,
        )

    # Profile legend (top center)
    graph_patches = []
    for pname, color, ax, _ in specs:
        sub = p1_radar[p1_radar[profile_col] == pname]
        if sub.empty:
            continue
        graph_patches.append(mpatches.Patch(facecolor=color, label=f'{pname} mean'))
    graph_patches.append(mpatches.Patch(facecolor='grey', alpha=0.35,
                                        label='IQR band (25th–75th pct.)'))

    leg0 = fig.legend(handles=graph_patches, loc='upper center',
                      bbox_to_anchor=(0.53, 0.90), ncol=1,
                      fontsize=FONT_SIZES['bottom_legends'], framealpha=0.95,
                      edgecolor='#CCCCCC',
                      title='Profiles  (mean line + IQR)',
                      title_fontsize=FONT_SIZES['legend_titles'])
    fig.add_artist(leg0)

    # Model family legend (bottom center)
    def _desc(m):
        return {'InT': 'Information Theory', 'StoP': 'Stochastic Processes',
                'DyS': 'Dynamical Systems',  'NeT':  'Network Theory',
                'StM': 'Statistical Mechanics'}.get(m, m)

    arc_patches = [mpatches.Patch(color=c, label=f'{m}  —  {_desc(m)}')
                   for m, c in MODEL_COLORS.items()]

    leg1 = fig.legend(handles=arc_patches, loc='lower center',
                      bbox_to_anchor=(0.50, -0.04), ncol=1,
                      fontsize=FONT_SIZES['bottom_legends'], framealpha=0.95,
                      edgecolor='#CCCCCC',
                      title='Model families  (axis color)',
                      title_fontsize=FONT_SIZES['legend_titles'])
    fig.add_artist(leg1)

    # Conceptual layer legend (bottom right)
    sec_patches = [mpatches.Patch(color=s['color'], alpha=0.55,
                                  label=s['name'])
                   for s in SECTION_LAYERS]
    fig.legend(handles=sec_patches, loc='lower right',
               bbox_to_anchor=(1.0, -0.04), ncol=1,
               fontsize=FONT_SIZES['bottom_legends'], framealpha=0.95,
               edgecolor='#CCCCCC',
               title='Conceptual layers  (band color)',
               title_fontsize=FONT_SIZES['legend_titles'])

    if save_path is not None:
        fig.savefig(save_path, bbox_inches='tight', dpi=300)
        print(f'Saved: {save_path}')

    plt.show()
    return fig, axes

In [ ]:
# Execute — Phase 1 Profile A vs B, balanced 5-axis radar
fig, axes = plot_phenotype_profiles_balanced(
    p1_pop=p1_pop,
    profile_col='profile_p1',
    participant_col='participant_id_canonical',
    save_path=FIG_OUT / 'fig4v3_balanced_phase1_radar.png',
)

## Summary

**Output:** `outputs/model_outputs/phenotyping/figures/fig4v3_balanced_phase1_radar.png`

**What to read from this chart:**
- If the NeT axis (Network concentration) still drives the A/B separation alone → confirms the
  two-layer structure finding (Layer 1 = network phenotype, Layer 2 = behavioral dynamics).
- If the other axes (InT, StoP, DyS, StM) show visible Profile A vs B differences → the
  original 7-axis chart was obscuring real differentiation by over-representing NeT.
- Either outcome is informative. This chart makes the family contribution honest and visible.

**Not done here:** new clustering, new features, any change to DS5_01 results.

**Next step (Level 2):** `notebooks/61_ds5_net_v2_no_selfloop.ipynb` — recompute NeT metrics
without same-app self-loops (DS5_02 Phase A Stage 1, A3). Blueprint first: `ds5_net_v2_blueprint.md`.